# I. 🎯 COHORT LÀ GÌ (1 câu đủ dùng phỏng vấn)

Cohort = nhóm user theo thời điểm bắt đầu (first activity) để theo dõi hành vi theo thời gian

# II. 🔧 BUILD AUTO COHORT
1. 📥 Create Cohort

In [ ]:
def create_cohort(df, customer_col, date_col):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # first purchase date (cohort)
    df["cohort_date"] = df.groupby(customer_col)[date_col].transform("min")

    # convert to month
    df["cohort_month"] = df["cohort_date"].dt.to_period("M")
    df["activity_month"] = df[date_col].dt.to_period("M")

    return df

df_cohort = create_cohort(df, "customer_id", "date")

2. 📊 Cohort Index (tháng thứ mấy)

In [ ]:
def add_cohort_index(df):
    df = df.copy()

    df["cohort_index"] = (
        (df["activity_month"].dt.year - df["cohort_month"].dt.year) * 12 +
        (df["activity_month"].dt.month - df["cohort_month"].dt.month)
    )

    return df

df_cohort = add_cohort_index(df_cohort)

3. 🔥 Retention Matrix

In [ ]:
def build_retention_matrix(df, customer_col):
    cohort_data = df.groupby(
        ["cohort_month", "cohort_index"]
    )[customer_col].nunique().reset_index()

    cohort_pivot = cohort_data.pivot(
        index="cohort_month",
        columns="cohort_index",
        values=customer_col
    )

    # normalize
    cohort_size = cohort_pivot.iloc[:,0]
    retention = cohort_pivot.divide(cohort_size, axis=0)

    return retention

retention_matrix = build_retention_matrix(df_cohort, "customer_id")
retention_matrix.head()

4. 📊 Visualization (Heatmap cực quan trọng)

In [ ]:
def plot_cohort(retention_matrix):
    plt.figure(figsize=(10,6))
    sns.heatmap(
        retention_matrix,
        annot=True,
        fmt=".0%",
        cmap="coolwarm"
    )
    plt.title("Cohort Retention Matrix")
    plt.show()

plot_cohort(retention_matrix)

👉 Đây là chart “signature” của Product Analyst

# III. 🧠 AUTO INSIGHT ENGINE (🔥)

In [ ]:
def cohort_insights(retention_matrix):
    insights = []

    # early drop (month 1)
    if 1 in retention_matrix.columns:
        avg_m1 = retention_matrix[1].mean()
        if avg_m1 < 0.3:
            insights.append("High early churn (Month 1 retention low)")

    # long-term retention
    if retention_matrix.shape[1] > 3:
        last_col = retention_matrix.columns[-1]
        avg_last = retention_matrix[last_col].mean()

        if avg_last < 0.1:
            insights.append("Very weak long-term retention")

    # improvement trend
    first_cohort = retention_matrix.iloc[0].mean()
    last_cohort = retention_matrix.iloc[-1].mean()

    if last_cohort > first_cohort:
        insights.append("New cohorts performing better")

    return insights

print(cohort_insights(retention_matrix))

# IV. 🔍 COHORT SEGMENTATION (🔥 rất mạnh)

In [ ]:
def cohort_by_segment(df, customer_col, segment_col):
    segments = df[segment_col].dropna().unique()

    for seg in segments:
        print(f"\n=== Segment: {seg} ===")

        temp = df[df[segment_col] == seg]
        retention = build_retention_matrix(temp, customer_col)

        plot_cohort(retention)

👉 Insight:

“Mobile cohort retention thấp hơn desktop”

# V. 🎯 REVENUE COHORT (ADVANCED)

👉 Không chỉ retention — mà còn tiền

In [ ]:
def revenue_cohort(df, customer_col, revenue_col):
    cohort_data = df.groupby(
        ["cohort_month", "cohort_index"]
    )[revenue_col].sum().reset_index()

    pivot = cohort_data.pivot(
        index="cohort_month",
        columns="cohort_index",
        values=revenue_col
    )

    return pivot

revenue_matrix = revenue_cohort(df_cohort, "customer_id", "order_value")

sns.heatmap(revenue_matrix, cmap="Blues")
plt.title("Revenue Cohort")
plt.show()

👉 Insight:

Cohort nào tạo nhiều tiền nhất

# VI. 🧠 STORYTELLING OUTPUT

👉 Ví dụ bạn nói trong interview:

What

Month 1 retention chỉ ~25%
Cohort mới cải thiện hơn cohort cũ

So what

User rời đi rất sớm → mất LTV

Now what

Optimize onboarding + early experience

# VII. 🔥 COMBINE VỚI CHURN & FUNNEL (LEVEL CAO)

👉 Khi bạn connect:

Cohort → thấy when churn happens
Funnel → thấy where churn happens
Churn → thấy who churns

👉 Insight level cao:

“User từ cohort tháng 3 churn mạnh ở ngày 7 do drop tại checkout trên mobile”

⚠️ GÓC THẲNG

Sai lầm phổ biến:

- Chỉ vẽ heatmap → không insight
- Không normalize → sai số
- Không segment